# Bone-Bioinformetics reproducible Colab

End-to-end run of the osteoporosis herb-module workflow on **real public data only** — no simulated pharmacology, expression, GWAS, network, perturbation or structure evidence. Steps that genuinely require full GWAS summary statistics, QTL tabix, large single-cell/spatial matrices or docking binaries are written as explicit `*_pending_steps.csv` records naming the required input.

**Evidence chain:** stable classical modules → component targets → osteoclast/BMSC dynamics → marker/spatial niche → human genetics → perturbation reversal → knowledge-graph + structure.

The notebook runs two scripts:
1. `scripts/nature_bone_pipeline.py` — core four-module pipeline (modules 1–4 of the main narrative).
2. `scripts/high_order_modules.py` — four advanced modules (genetics-anchored causal pharmacology, single-cell/spatial niche, perturbation reversal, knowledge graph + structure).

## 1. Get the repository
If you opened this notebook from GitHub/Colab the files may already be present; otherwise set `REPO_URL` to your fork.

In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/psknlr/Bone-Bioinformetics.git'  # replace with your fork if needed
if not Path('scripts/nature_bone_pipeline.py').exists():
    !git clone {REPO_URL} repo
    %cd repo
else:
    print('Repository files already available.')

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt

## 3. Run the core pipeline (main modules 1–4)
The first run downloads/caches PubChem, ChEMBL, Open Targets, STRING, PanglaoDB, GEO (GSE224152, GSE246769) and GWAS Catalog resources. Cached under `.cache/` for repeatability.

In [ ]:
!python scripts/nature_bone_pipeline.py --workbook osteoporosis_extraction_output_finalV6.xlsx --outdir results --analysis-scope full

## 4. Run the four advanced high-order modules
Reads `results/tables/` from step 3. Real API computation (Open Targets genetic datatypes, MyGene, eQTL Catalogue, Enrichr LINCS, Reactome, UniProt, AlphaFold, PDB) where available; heavy steps recorded as pending.

In [ ]:
!python scripts/high_order_modules.py --outdir results --modules 1,2,3,4

## 5. Module 1 — stable classical herb modules and historical stability
Multiple structured extractions of the same source passage are reconciled (default `merge` = expert-confirmed union), not silently de-duplicated to the first row. `dedup_strategy_sensitivity.csv` shows the core-quartet count under each strategy and `dedup_conflict_log.csv` records field-level conflicts. `core_module_significance.csv` tests the 杜仲–牛膝–续断–骨碎补 quartet and **every** sub-combination: under proper reconciliation the quartet **does co-occur** (in 《病机沙篆》, support = 1, lift ≈ 300×, FDR < 0.01) — rare but real, not the keep-first "support = 0" artifact.

In [ ]:
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
tables = Path('results/tables')
def show(name, n=12):
    p = tables/name
    if p.exists():
        print('\n###', name); display(pd.read_csv(p).head(n))
    else:
        print('\n###', name, '(not found)')
for name in ['dedup_strategy_sensitivity.csv','dedup_conflict_log.csv','core_module_significance.csv','stable_herb_modules.csv','historical_stability.csv']:
    show(name)

## 6. Module 2 — component targets and network medicine
Chemical-identity audit (PubChem CID/InChIKey, ChEMBL InChIKey, synonym-collision flags), **single-protein** targets only (cell-line/phenotypic assays excluded), activities de-duplicated and aggregated per gene, and a **STRING physical-interaction** network. Proximity is scored with a degree-preserving random reference (z-score, empirical p) plus a size-matched null from an independent background.

In [ ]:
for name in ['pubchem_chembl_compounds.csv','component_target_evidence_tiers.csv','component_target_summary.csv','excluded_nonmolecular_activities.csv','predicted_target_layer.csv','network_proximity.csv','network_proximity_random_combo_null.csv']:
    show(name)

## 7. Module 3 — reference-marker / expression localisation
Marker overlap with hypergeometric enrichment, real GSE224152 marrow expression and GSE246769 osteoclast-differentiation dynamics.

In [ ]:
for name in ['reference_marker_overlap_localisation.csv','single_cell_localisation.csv','gse224152_target_expression.csv','gse246769_osteoclast_dynamics.csv']:
    show(name)

## 8. Module 4 — human genetics and four-pillar convergence
Gene-level GWAS-Catalog enrichment and the classical/pharmacology/cell-expression/human-genetics convergence table.

In [ ]:
for name in ['gwas_gene_enrichment.csv','gwas_catalog_bone_trait_studies.csv','human_genetics_prioritised_targets.csv','genetics_anchored_causal_pharmacology_scores.csv']:
    show(name)

## 9. High-order Module 1 — genetics-anchored multi-omics causal pharmacology
Open Targets `genetic_association` datatype scores (the real human-genetics anchor), eQTL presence, and A/B/C grading where A requires real human-genetics support. Fine-mapping/coloc/SMR/MR remain pending on full summary statistics.

In [ ]:
for name in ['m1_genetics_anchored_causal_targets.csv','m1_opentargets_genetic_datatypes.csv','m1_causal_pending_steps.csv']:
    show(name)

## 10. High-order Module 2 — single-cell / spatial bone-niche localisation
Marker-niche assignment (osteoprogenitor / osteoclast-immune / vascular-osteogenic / marrow-adipogenic) and real GSE246769 module-expression scores. scVI/cell2location/CellChat/trajectory pending on large matrices.

In [ ]:
for name in ['m2_niche_localisation.csv','m2_module_expression_scores.csv','m2_pending_steps.csv']:
    show(name)

## 11. High-order Module 3 — perturbation-omics disease-signature reversal
Osteoclast up/down signature from GSE246769 queried against Enrichr LINCS L1000 chemical and CRISPR-KO consensus libraries. Level-5 weighted CMap and scPerturb E-distance pending.

In [ ]:
for name in ['m3_perturbation_reversal.csv','m3_pending_steps.csv']:
    show(name, n=15)

## 12. High-order Module 4 — knowledge graph + structural pharmacology
Heterogeneous herb→compound→target→pathway→disease graph with Reactome + Open Targets edges, shared-pathway meta-path link prediction, and real UniProt/AlphaFold/PDB structure availability. PyKEEN/R-GCN, docking and MD pending on binaries.

In [ ]:
for name in ['m4_knowledge_graph_edges.csv','m4_link_prediction.csv','m4_structure_evidence.csv','m4_pending_steps.csv']:
    show(name)

## 13. Display generated figures
PNG figures are generated at runtime (ignored by git) and shown here.

In [ ]:
from IPython.display import Image, display
figdir = Path('results/figures')
figs = sorted(figdir.glob('Fig*.png'))
if not figs:
    print('No figures found — run step 3 first.')
for fig in figs:
    print(fig.name); display(Image(filename=str(fig)))